In [1]:
!pip install langchain langchain_huggingface langchain chromadb faiss-cpu tiktoken langchain_community wikipedia

  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 16.0 which is incompatible.
gradio-client 1.14.0 requires websockets<16.0,>=13.0, but you have websockets 16.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.


### Wikipedia Retriever

In [2]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k=2, language='en')

In [3]:
query = "The history of mughal empires."

docs = retriever.invoke(query)

In [4]:
docs[0].page_content

'The Mughal Empire was an early modern empire in South Asia. At its peak, the empire stretched from the outer fringes of the Indus River Basin in the west, northern Afghanistan in the northwest, and Kashmir in the north, to the highlands of present-day Assam and Bangladesh in the east, and the uplands of the Deccan Plateau in South India.\nThe Mughal Empire is conventionally said to have been founded in 1526 by Babur, a ruler from what is now Uzbekistan, who with the help of the neighbouring Safavid and Ottoman Empires, defeated the sultan of Delhi, Ibrahim Lodi, in the First Battle of Panipat and swept down the plains of North India. The Mughal imperial structure, however, is sometimes dated to 1600, to the rule of Babur\'s grandson, Akbar. This imperial structure lasted until 1720, shortly after the death of the last major emperor, Aurangzeb, during whose reign the empire also achieved its maximum geographical extent. Reduced subsequently to the region in and around Old Delhi by 1760

### Vector Stores

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily"),
    Document(page_content="Chroma is a vector database optimized for LLM-based search"),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models. ")
]

In [8]:
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_function,
    collection_name = "my_collection"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [10]:
query = "What is Croma used for?"
result = retriever.invoke(query)

result

[Document(metadata={}, page_content='Chroma is a vector database optimized for LLM-based search'),
 Document(metadata={}, page_content='Embeddings convert text into high-dimensional vectors.')]

### Maximal Marginal Retriever (MMR)

In [11]:
documents = [
    Document(page_content="LangChain makes it eay to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR hellps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more.")
]

In [12]:
from langchain_community.vectorstores import FAISS

embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(
    documents = documents,
    embedding = embedding_function
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs={'top':3, 'lambda_mult': 0.5}
)

In [14]:
query = "What is LangChain?"

result = retriever.invoke(query)

result

[Document(id='1ec31ff4-f1bd-449f-991c-cf38fef3fd35', metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'),
 Document(id='fd72d2ba-6a9a-4ed7-8802-9f834a160730', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='265fcff9-6f5d-4ced-804c-2765973f500e', metadata={}, page_content='Embeddings are vector representations of text.'),
 Document(id='64dbdc70-6564-4db4-84ec-c803d24a521e', metadata={}, page_content='MMR hellps you get diverse results when doing similarity search.')]

### Multi-Query Retriever

In [15]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
# from langchain_community.retrievers.multi_query import MultiQueryRetriever

In [17]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [18]:
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embedding_function
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
"""retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(search_kwargs={'k':5}),
    llm = model
)"""

"retriever = MultiQueryRetriever.from_llm(\n    retriever = vectorstore.as_retriever(search_kwargs={'k':5}),\n    llm = model\n)"

In [20]:
query = "How to improve energy level and maintain balance?"

result = retriever.invoke(query)

result

[Document(id='1ec31ff4-f1bd-449f-991c-cf38fef3fd35', metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'),
 Document(id='64dbdc70-6564-4db4-84ec-c803d24a521e', metadata={}, page_content='MMR hellps you get diverse results when doing similarity search.'),
 Document(id='265fcff9-6f5d-4ced-804c-2765973f500e', metadata={}, page_content='Embeddings are vector representations of text.'),
 Document(id='fd72d2ba-6a9a-4ed7-8802-9f834a160730', metadata={}, page_content='LangChain is used to build LLM based applications.')]

### Contextual Compression Retriever

In [23]:
from langchain_community.retrievers import ContextualCompressionRetriever
from langchain_community.retrievers import LLMChainExtractor

ImportError: cannot import name 'ContextualCompressionRetriever' from 'langchain_community.retrievers' (/usr/local/lib/python3.12/dist-packages/langchain_community/retrievers/__init__.py)

In [ ]:
from langchain_community.retrievers import ContextualCompressionRetriever

In [ ]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding = embedding_function
)

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={'k':5})

In [ ]:
compressor = LLMChainExtractor.from_llm(model)

In [ ]:
compressor_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [ ]:
query = "What is photosynthesis?"
result = retriever.invoke()
result